# Project 1 
## Exploratory Data Analysis

In [1]:
# Importing Libraries
import pandas as pd
import numpy as np


In [2]:
# Defining dataset paths
movies_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\movies.csv"
cast_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\cast.csv"
crew_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\crew.csv"
genres_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\genres.csv"
movie_genres_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\movie_genres.csv"
movie_keywords_path = r"C:\Users\shubham.saini\Desktop\Data science\Projects\Project 1\Data\movie_keywords.csv"

# Loading Dataset
movies = pd.read_csv(movies_path)
cast = pd.read_csv(cast_path)
crew = pd.read_csv(crew_path)
genres = pd.read_csv(genres_path)
movie_genres = pd.read_csv(movie_genres_path)
movie_keywords = pd.read_csv(movie_keywords_path)

tables = {"movies" : movies, "cast": cast, "crew": crew, "genres": genres, "movie_genres": movie_genres, "movie_keywords": movie_keywords
}


In [3]:
# Checking shape, data types, nulls, duplicates
for name, df in tables.items():
    print("=" * 60)
    print(f"Filename: {name} -> Rows: {df.shape[0]} , Columns: {df.shape[1]}")
    print("=" * 60)
    print(f"Column Datatypes:\n{df.dtypes}")
    print(f"\nNull Counts:\n{df.isna().sum()}\n")
    print(f"Duplicate Rows: {df.duplicated().sum()}")


Filename: movies -> Rows: 2509 , Columns: 12
Column Datatypes:
movie_id               int64
title                    str
original_language        str
release_date             str
runtime                int64
budget               float64
revenue              float64
popularity           float64
vote_average         float64
vote_count             int64
status                   str
overview                 str
dtype: object

Null Counts:
movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64

Duplicate Rows: 6
Filename: cast -> Rows: 24917 , Columns: 6
Column Datatypes:
cast_row_id       int64
movie_id          int64
person_id         int64
actor_name          str
character_name      str
cast_order        int64
dtype: object

Null Counts:
cast_row_id        0

In [4]:
# checking first 3 rows of each file
for name, df in tables.items():
    print(f"--------------- {name} -------------------")
    print(df.head(3))

--------------- movies -------------------
   movie_id         title original_language release_date  runtime  \
0    157336  Interstellar                en   2014-11-05      169   
1     27205     Inception                en   2010-07-15      148   
2     24428  The Avengers                en   2012-04-25      143   

        budget       revenue  popularity  vote_average  vote_count    status  \
0  165000000.0  7.466067e+08     73.0452         8.484       40566  Released   
1  160000000.0  8.390306e+08     55.8497         8.400       39727  Released   
2  220000000.0  1.518816e+09     72.2515         8.062       38993  Released   

                                            overview  
0  The adventures of a group of explorers who mak...  
1  Cobb, a skilled thief who commits corporate es...  
2  When an unexpected enemy emerges and threatens...  
--------------- cast -------------------
   cast_row_id  movie_id  person_id           actor_name   character_name  \
0            1    157

In [5]:
# Cleaning movies table and keeping logs
cleaning_log = []                       
rows_before = len(movies)

# Remove exact duplicate rows
movies = movies.drop_duplicates().reset_index(drop=True)

# Clean text columns: remove extra spaces
for col in ["title", "original_language", "status", "overview"]:
    movies[col] = movies[col].str.strip()

# Enforce data types
movies["movie_id"] = movies["movie_id"].astype(int)
movies["release_date"] = pd.to_datetime(movies["release_date"], errors="coerce")

# Logs
cleaning_log.append({"table": "movies", "rows_before": rows_before, "rows_after": len(movies)})

# Checks
print("movie_id unique:", movies["movie_id"].is_unique)
print(f"Rows: {rows_before} -> {len(movies)}")
print("\nNulls after cleaning:")
print(movies.isna().sum())
print("\nDtypes:")
print(movies.dtypes)

movie_id unique: True
Rows: 2509 -> 2503

Nulls after cleaning:
movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64

Dtypes:
movie_id                      int64
title                           str
original_language               str
release_date         datetime64[us]
runtime                       int64
budget                      float64
revenue                     float64
popularity                  float64
vote_average                float64
vote_count                    int64
status                          str
overview                        str
dtype: object


In [6]:
#  Cleaning Genre
rows_before = len(genres)

# Remove extra spaces in names and drop exact duplicates
genres["genre_name"] = genres["genre_name"].str.strip()
genres = genres.drop_duplicates().reset_index(drop=True)

# Enforce data types
genres["genre_id"] = genres["genre_id"].astype(int)

cleaning_log.append({"table": "genres", "rows_before": rows_before, "rows_after": len(genres)})

# Check: each genre_id must appear once
print("genre_id unique:", genres["genre_id"].is_unique)
print(f"Rows: {rows_before} -> {len(genres)}")

genre_id unique: True
Rows: 19 -> 19


In [7]:
# Cleaning movie_genres 
rows_before = len(movie_genres)

# 1) Remove exact duplicate rows
movie_genres = movie_genres.drop_duplicates()

# 2) Keep only rows whose movie_id and genre_id exist in the parent tables
movie_genres = movie_genres[
    movie_genres["movie_id"].isin(movies["movie_id"]) &
    movie_genres["genre_id"].isin(genres["genre_id"])
].reset_index(drop=True)

cleaning_log.append({"table": "movie_genres", "rows_before": rows_before, "rows_after": len(movie_genres)})
print(f"Rows: {rows_before} -> {len(movie_genres)}")

Rows: 6888 -> 6878


In [8]:
# Cleaning cast
rows_before = len(cast)

# Remove duplicate rows
cast = cast.drop_duplicates()

# Keep only rows where movie_id exists in the movies table
cast = cast[cast["movie_id"].isin(movies["movie_id"])]

# Remove extra spaces in names; empty text becomes a real null
cast["actor_name"] = cast["actor_name"].str.strip()
cast["character_name"] = cast["character_name"].str.strip().replace("", None)

cast = cast.reset_index(drop=True)
cleaning_log.append({"table": "cast", "rows_before": rows_before, "rows_after": len(cast)})
print(f"Rows: {rows_before} -> {len(cast)}")
print("Null character_name:", cast["character_name"].isnull().sum())



Rows: 24917 -> 24902
Null character_name: 11


In [9]:
# Cleaning crew
rows_before = len(crew)

crew = crew.drop_duplicates()
crew = crew[crew["movie_id"].isin(movies["movie_id"])]

# Remove extra spaces in text columns
crew["person_name"] = crew["person_name"].str.strip()
crew["job"] = crew["job"].str.strip()
crew["department"] = crew["department"].str.strip()

crew = crew.reset_index(drop=True)
cleaning_log.append({"table": "crew", "rows_before": rows_before, "rows_after": len(crew)})
print(f"Rows: {rows_before} -> {len(crew)}")

Rows: 7297 -> 7289


In [10]:
# Cleaning movie_keywords
rows_before = len(movie_keywords)

movie_keywords = movie_keywords.drop_duplicates()
movie_keywords = movie_keywords[movie_keywords["movie_id"].isin(movies["movie_id"])]

# Remove extra spaces in keyword text
movie_keywords["keyword_name"] = movie_keywords["keyword_name"].str.strip()

movie_keywords = movie_keywords.reset_index(drop=True)
cleaning_log.append({"table": "movie_keywords", "rows_before": rows_before, "rows_after": len(movie_keywords)})
print(f"Rows: {rows_before} -> {len(movie_keywords)}")

Rows: 41701 -> 41681


In [11]:
cleaning_log

[{'table': 'movies', 'rows_before': 2509, 'rows_after': 2503},
 {'table': 'genres', 'rows_before': 19, 'rows_after': 19},
 {'table': 'movie_genres', 'rows_before': 6888, 'rows_after': 6878},
 {'table': 'cast', 'rows_before': 24917, 'rows_after': 24902},
 {'table': 'crew', 'rows_before': 7297, 'rows_after': 7289},
 {'table': 'movie_keywords', 'rows_before': 41701, 'rows_after': 41681}]

## SQL Connection

In [12]:
!pip3 install mysql-connector-python


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
# Connecting to MySQL 
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",                    
    password="Assasin#96"         
)
cursor = conn.cursor()

# Creating database
cursor.execute("CREATE DATABASE IF NOT EXISTS project_1")
cursor.execute("USE project_1")
print("Connected. Using database: project_1")

Connected. Using database: project_1


In [14]:
# Creating tables in Mysql
for t in ["movie_keywords", "crew", "cast", "movie_genres", "genres", "movies"]:
    cursor.execute(f"DROP TABLE IF EXISTS {t}")

cursor.execute("""
CREATE TABLE movies (
    movie_id          INT PRIMARY KEY,
    title             VARCHAR(255),
    original_language VARCHAR(10),
    release_date      DATE,
    runtime           INT,
    budget            BIGINT,
    revenue           BIGINT,
    popularity        FLOAT,
    vote_average      FLOAT,
    vote_count        INT,
    status            VARCHAR(50),
    overview          TEXT
)""")

cursor.execute("""
CREATE TABLE genres (
    genre_id   INT PRIMARY KEY,
    genre_name VARCHAR(50)
)""")

cursor.execute("""
CREATE TABLE movie_genres (
    movie_id INT,
    genre_id INT,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id),
    FOREIGN KEY (genre_id) REFERENCES genres(genre_id)
)""")

cursor.execute("""
CREATE TABLE cast (
    cast_row_id    INT PRIMARY KEY AUTO_INCREMENT,
    movie_id       INT,
    person_id      INT,
    actor_name     VARCHAR(255),
    character_name VARCHAR(500),
    cast_order     INT,
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)""")

cursor.execute("""
CREATE TABLE crew (
    crew_row_id INT PRIMARY KEY AUTO_INCREMENT,
    movie_id    INT,
    person_id   INT,
    person_name VARCHAR(255),
    job         VARCHAR(100),
    department  VARCHAR(100),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)""")

cursor.execute("""
CREATE TABLE movie_keywords (
    movie_id     INT,
    keyword_id   INT,
    keyword_name VARCHAR(100),
    FOREIGN KEY (movie_id) REFERENCES movies(movie_id)
)""")

cursor.execute("SHOW TABLES")
print("Tables created:", [row[0] for row in cursor.fetchall()])

Tables created: ['cast', 'crew', 'genres', 'movie_genres', 'movie_keywords', 'movies']


In [ ]:
# Loading data in tables

movies["budget"]  = movies["budget"].astype("int64")
movies["revenue"] = movies["revenue"].astype("int64")

def load_table(table_name, df):
    # converting missing value (NaN / NaT) into None, which MySQL stores as NULL
    data = df.astype(object).where(df.notna(), None).values.tolist()
    columns = ", ".join(df.columns)                    
    placeholders = ", ".join(["%s"] * len(df.columns)) 
    cursor.executemany(f"INSERT INTO {table_name} ({columns}) VALUES ({placeholders})", data)
    conn.commit()                                      
    print(f"Loaded {table_name}: {len(df)} rows")

# Parents first, children after
load_table("movies", movies)
load_table("genres", genres)
load_table("movie_genres", movie_genres)
load_table("cast", cast)
load_table("crew", crew)
load_table("movie_keywords", movie_keywords)

Loaded movies: 2503 rows
Loaded genres: 19 rows
Loaded movie_genres: 6878 rows
Loaded cast: 24902 rows
Loaded crew: 7289 rows
Loaded movie_keywords: 41681 rows


In [25]:
# Step 7b: Compare rows in pandas with rows in MySQL (they must match)
tables = {"movies": movies, "genres": genres, "movie_genres": movie_genres,
          "cast": cast, "crew": crew, "movie_keywords": movie_keywords}

for name, df in tables.items():
    cursor.execute(f"SELECT COUNT(*) FROM {name}")
    sql_rows = cursor.fetchone()[0]
    print(f"{name:<15} pandas: {len(df):>6}   mysql: {sql_rows:>6}   {'OK' if len(df) == sql_rows else 'MISMATCH'}")

movies          pandas:   2503   mysql:   2503   OK
genres          pandas:     19   mysql:     19   OK
movie_genres    pandas:   6878   mysql:   6878   OK
cast            pandas:  24902   mysql:  24902   OK
crew            pandas:   7289   mysql:   7289   OK
movie_keywords  pandas:  41681   mysql:  41681   OK


In [27]:
# Closing the connection
cursor.close()
conn.close()
print("Connection closed.")

Connection closed.
